In [1]:
import sys

import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# EEGNet-specific imports
from EEGModels import EEGNet
from tensorflow.keras import utils as np_utils
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import backend as K

import tensorflow.keras as keras
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
import tensorflow.keras.models as models
import tensorflow.compat.v1 as tf
import numpy as np
from tslearn.metrics import soft_dtw
import os
import mne
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Permute, Dropout, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import SeparableConv2D, DepthwiseConv2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import SpatialDropout2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.layers import Input, Flatten
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
from scipy.signal import butter, filtfilt
from sklearn.model_selection import KFold
from scipy import io, signal
import math
import json

In [2]:
n_targets = 10

In [3]:
def extract_accuracies_from_file(file_path):
    # Open and load the JSON file
    with open(file_path, 'r') as file:
        json_data = json.load(file)
    
    # Extract the accuracies into a list
    accuracies = [data[0].get("accuracy", None) for data in json_data.values()]
    return accuracies

In [4]:
def calculate_itr(classification_acc, num_targets, target_selection_time_seconds):          
    p = classification_acc
    logp = np.log2(p)
    ip = 1.0 - p

    if ip == 0:
        ip += 0.0001

    a = np.log2(num_targets)
    b = p * logp
    c = ip * np.log2( ip/(num_targets-1) )
    result = a + b + c
    return result * (60 / target_selection_time_seconds)

In [5]:
# EEGNet 8 - 90Hz 数据长度 1 - 0.4s
eeg_8_250 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_results_9chans_8_90_250.json")
print(np.mean(eeg_8_250))
print(np.std(eeg_8_250))

eeg_8_200 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_results_9chans_8_90_200.json")
print(np.mean(eeg_8_200))
print(np.std(eeg_8_200))

eeg_8_150 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_results_9chans_8_90_150.json")
print(np.mean(eeg_8_150))
print(np.std(eeg_8_150))

eeg_8_100 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_results_9chans_8_90_100.json")
print(np.mean(eeg_8_100))
print(np.std(eeg_8_100))

0.7678571437086378
0.20244676999548736
0.7537142864295414
0.21072779398082234
0.7042857191392353
0.1869276644808794
0.6162857085466384
0.18431472436654067


In [6]:
# for i in eeg_8_200:
#     print(i)

In [7]:
itr_eeg = []
t = 0.8
for i in eeg_8_200:
    itr = calculate_itr(i, n_targets, t)
#    print(itr)
    itr_eeg.append(itr)
print(np.mean(itr_eeg))
print(np.std(itr_eeg)/3)

142.94434389809496
21.466263801705505


In [8]:
# FB-EEGNet 8, 16, 24 - 90Hz 数据长度 1 - 0.4s
eeg_fusion3_250 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_fusion3_results_9chans_90_250.json")
print(np.mean(eeg_fusion3_250))
print(np.std(eeg_fusion3_250))

eeg_fusion3_200 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_fusion3_results_9chans_90_200.json")
print(np.mean(eeg_fusion3_200))
print(np.std(eeg_fusion3_200))

eeg_fusion3_150 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_fusion3_results_9chans_90_150.json")
print(np.mean(eeg_fusion3_150))
print(np.std(eeg_fusion3_150))

eeg_fusion3_100 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/eeg_fusion3_results_9chans_90_100.json")
print(np.mean(eeg_fusion3_100))
print(np.std(eeg_fusion3_100))

0.7714285722800663
0.17779833094779635
0.684571430512837
0.1767054472507221
0.6495238150869097
0.16268997506396232
0.5991428524255753
0.18266865303104524


In [9]:
itr_fb = []
t = 1.0
for i in eeg_fusion3_250:
    itr = calculate_itr(i, n_targets, t)
#    print(itr)
    itr_eeg.append(itr)
print(np.mean(itr_eeg))
print(np.std(itr_eeg)/3)

129.93607637101834
19.32337766606786


In [10]:
# CCA 8 - 90Hz 数据长度 1 - 0.4s
cca_8_250 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/cca_results_9chan_8_90_1.0s.json")
print(np.mean(cca_8_250))
print(np.std(cca_8_250))

cca_8_200 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/cca_results_9chan_8_90_0.8s.json")
print(np.mean(cca_8_200))
print(np.std(cca_8_200))

cca_8_150 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/cca_results_9chan_8_90_0.6s.json")
print(np.mean(cca_8_150))
print(np.std(cca_8_150))

cca_8_100 = extract_accuracies_from_file("D:/2024/7-9/dissertation/results/results/cca_results_9chan_8_90_0.4s.json")
print(np.mean(cca_8_100))
print(np.std(cca_8_100))

52.47619047619048
25.446400027831388
44.142857142857146
22.542275728314078
31.238095238095237
20.649543048646716
19.380952380952376
12.50451165745321


In [19]:
itr_cca = []
t = 1.0
for i in cca_8_250:
    itr = calculate_itr(i/100, n_targets, t)
#    print(itr)
    itr_cca.append(itr)
print(np.mean(itr_cca))
print(np.std(itr_cca)/3)

61.37473080764956
17.334994235229775
